In [10]:
import requests
import pandas as pd
import requests

In [4]:
# Feed GBFS principal
url = "https://gbfs.mex.lyftbikes.com/gbfs/es/station_information.json"

response = requests.get(url)
response.raise_for_status()

data = response.json()

stations = data["data"]["stations"]

df = pd.DataFrame(stations)
col_sel = ["station_id", "short_name", "name","lat","lon","capacity"]
df = df[col_sel]
print(df.head())

# df.to_excel(
#     "ecobici_estaciones.xlsx",
#     index=False
# )

print(f"Total estaciones: {len(df)}")

  station_id short_name                                               name  \
0          1        710     CE-710 Molino del Rey - Glorieta de la Lealtad   
1          5        407  CE-407  Prolongación Xochicalco-General Emilia...   
2          6        428  CE-428  Prolongación Uxmal-Av. Popocatépetl (E...   
3          7        483         CE-483 Colegio Salesiano - Marina Nacional   
4          8        443                    CE-443 Bruno Traven-Golondrinas   

         lat        lon  capacity  
0  19.416795 -99.192508        39  
1  19.367266 -99.158656        19  
2  19.363404 -99.160395        27  
3  19.443040 -99.177590         0  
4  19.359583 -99.162085        31  
Total estaciones: 681


In [3]:
df.head()

,station_id,external_id,name,short_name,lat,lon,rental_methods,capacity,electric_bike_surcharge_waiver,is_charging,eightd_has_key_dispenser,has_kiosk
0,1,e961269c-34c4-4b70-8e30-a51aa95a8429,CE-710 Molino del Rey - Glorieta de la Lealtad,710,19.416795,-99.192508,"[CREDITCARD, KEY]",39,False,False,False,True
1,5,3ea89109-d2f3-46eb-9c41-c43742050340,CE-407 Prolongación Xochicalco-General Emilia...,407,19.367266,-99.158656,"[CREDITCARD, KEY]",19,False,False,False,True
2,6,ba78b703-4e5a-44bd-ab2c-1eedc71e11c3,CE-428 Prolongación Uxmal-Av. Popocatépetl (E...,428,19.363404,-99.160395,"[CREDITCARD, KEY]",27,False,False,False,True
3,7,6563d263-2342-46e3-8983-461e68d2d615,CE-483 Colegio Salesiano - Marina Nacional,483,19.443040,-99.177590,"[CREDITCARD, KEY]",0,False,False,False,True
4,8,ec55e597-c8fc-4e86-bcfe-b0e81a494790,CE-443 Bruno Traven-Golondrinas,443,19.359583,-99.162085,"[CREDITCARD, KEY]",31,False,False,False,True


In [7]:
BASE_URL = "https://ecobici.cdmx.gob.mx/"
DATA_URL = "https://ecobici.cdmx.gob.mx/en/open-data/"
GBFS_URL = "https://gbfs.mex.lyftbikes.com/gbfs/es/station_information.json"

In [8]:
# descarga de estaciones
def download_stations():
    # selección campos
    col_sel = ["station_id", "short_name", "name","lat","lon","capacity"]
    # request
    response = requests.get(GBFS_URL)
    response.raise_for_status()
    data = response.json()
    stations = data["data"]["stations"]
    # dataframe
    df = pd.DataFrame(stations)
    df = df[col_sel]
    return df

In [11]:
print(f"📥 Descargando: dim_station")
df_stations = download_stations()
df_stations

📥 Descargando: dim_station


,station_id,short_name,name,lat,lon,capacity
0,1,710,CE-710 Molino del Rey - Glorieta de la Lealtad,19.416795,-99.192508,39
1,5,407,CE-407 Prolongación Xochicalco-General Emilia...,19.367266,-99.158656,19
2,6,428,CE-428 Prolongación Uxmal-Av. Popocatépetl (E...,19.363404,-99.160395,27
3,7,483,CE-483 Colegio Salesiano - Marina Nacional,19.443040,-99.177590,0
4,8,443,CE-443 Bruno Traven-Golondrinas,19.359583,-99.162085,31
...,...,...,...,...,...,...
676,702,524,CE-524 Malvón - Juan Sarabia,19.468119,-99.167268,23
677,703,Temporal 2,Temporal (Claz. Tlalpan - Cjon del esfuerzo),19.309943,-99.141642,3
678,704,Temporal 4,Temporal (Calz Tlalpan - Esq. Nadadores),19.350244,-99.145295,3
679,720,Temporal 1,Temporal (Anillo de Circunvalación - Calz de T...,19.340205,-99.143613,3


In [26]:
def transform_dim_station(base):
    df = base.copy()
    df["station_id"] = pd.to_numeric(df["station_id"], errors="coerce")
    df["short_name"] = df["short_name"].astype(str)
    df["name"] = df["name"].astype(str)
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce").round(7)
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce").round(7)
    df["capacity"] = pd.to_numeric(df["capacity"], errors="coerce").fillna(0)
    return df

In [27]:
df_transform_stations = transform_dim_station(df_stations)


In [15]:
def load_data(db, data, table_name):

    if data.empty:
        return
    # carga a base de datos
    db.insert_to_db(data, tabla=table_name, batch_size=500)

In [19]:
import os
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "https://ecobici.cdmx.gob.mx/"
DATA_URL = "https://ecobici.cdmx.gob.mx/en/open-data/"
GBFS_URL = "https://gbfs.mex.lyftbikes.com/gbfs/es/station_information.json"

DATA_PATH = "data/raw/"

# configuración base de datos
class DB:
    HOST = os.getenv("DB_HOST")
    PORT = os.getenv("DB_PORT")
    NAME = os.getenv("DB_NAME")
    USER = os.getenv("DB_USER")
    PASSWORD = os.getenv("DB_PASSWORD")
    
MYSQL_CONFIG = {
    "database": "ecobicis"
}

In [23]:

# Clase 
class MySQLDatabase():
    def __init__(self, database):
        self.host = "localhost"
        self.user = "root"
        self.password = "astro123"
        self.database = database
        self.connection = None

        # Crear conexión al instanciar
        self.connect()
        
    # método para establecer conexión
    def connect(self):
        host = self.host
        database = self.database
        username = self.user
        password = self.password
        # crea la conexión si no existe
        if self.connection and self.connection.is_connected():
            return
    
        try:
            self.connection = mysql.connector.connect(host=host, 
                                            database=database, 
                                            user=username, 
                                            password=password)
            print("✅ Conexión exitosa")
        except Error as e:
            print("Error: ",e)
            self.connection = None
    
    # método para ejecutar escript, no retorna nada
    def execute(self, query):
        """Ejecuta queries tipo DDL (CREATE, DROP, ALTER)."""
        #self.connect()
        conn = self.connection
        cursor = conn.cursor()

        try:
            cursor.execute(query)
            conn.commit()
            #print("✅ Query ejecutado correctamente")
        except Error as e:
            print(f"Error ejecutando query: {e}")
        finally:
            cursor.close()
    
    # método para ejecutar un query
    def execute_query(self, query, values=None):
        """Ejecuta una consulta SQL y retorna un DataFrame."""
        #self.connect()
        try:
            df = pd.read_sql(query, con=self.connection, params=values)
            return df
        except Error as e:
            print(f"Error ejecutando query: {e}")
            return None

    # método par cargar datos a una tabla
    def insert_to_db(self, df, tabla, batch_size=5000):
        """
        Inserta un DataFrame en MySQL evitando duplicados (UPSERT).
        """

        #self.connect()
        #conn = self.connection
        try:
            cursor = self.connection.cursor()

            # Reemplazar NaN con None
            df = df.where(pd.notnull(df), None)

            # Convertir DataFrame a lista de tuplas
            values = [tuple(row) for row in df.itertuples(index=False, name=None)]

            # Columnas
            cols = ", ".join(df.columns)

            # Placeholders
            placeholders = ", ".join(["%s"] * len(df.columns))

            # Construir UPDATE dinámico (excluyendo id si existiera)
            update_cols = ", ".join([
                f"{col}=VALUES({col})"
                for col in df.columns
            ])

            # Query tipo UPSERT
            insert_query = f"""
            INSERT INTO {tabla} ({cols})
            VALUES ({placeholders})
            ON DUPLICATE KEY UPDATE
            {update_cols}
            """

            # Inserción por lotes
            for start in range(0, len(values), batch_size):
                end = start + batch_size
                cursor.executemany(insert_query, values[start:end])
                self.connection.commit()
        except Error as e:
            print(f"❌ Error insertando datos: {e}")

        #finally:
        #    cursor.close()


    def close(self):
        """Cierra la conexión."""
        if self.connection and self.connection.is_connected():
            self.connection.close()
            print("🔒 Conexión cerrada")
            self.connection = None

In [24]:
db = MySQLDatabase("ecobicis")

✅ Conexión exitosa


In [32]:
df_transform_stations.sort_values('short_name')

,station_id,short_name,name,lat,lon,capacity
284,292,001,CE-001 Río Sena-Río Balsas,19.433785,-99.167987,35
136,142,002,CE-002 Río Guadalquivir - Río Nazas,19.430495,-99.171160,23
107,113,003,CE-003 Reforma - Insurgentes,19.431630,-99.158547,31
454,465,004,CE-004 Río Nilo - Río Panuco,19.428491,-99.171693,19
117,123,005,CE-005 Río Pánuco Río Tiber,19.429843,-99.169370,23
...,...,...,...,...,...,...
675,700,711,CE-711 Molino del Rey - Av. Constituyentes,19.414108,-99.191983,39
679,720,Temporal 1,Temporal (Anillo de Circunvalación - Calz de T...,19.340205,-99.143613,3
677,703,Temporal 2,Temporal (Claz. Tlalpan - Cjon del esfuerzo),19.309943,-99.141642,3
680,721,Temporal 3,Temporal (Claz. Tlalpan - Cerro San Antonio),19.339392,-99.142743,3


In [28]:
load_data(db, df_transform_stations, table_name='dim_station')